# 🧪 Optuna ile Hiperparametre Optimizasyonu (Kaggle Master Stili)

GridSearch veya RandomSearch yerine Bayesian yaklaşımıyla çalışan Optuna'nın en verimli kullanım şablonu.

In [ ]:
import optuna
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
import numpy as np

## LightGBM İçin Optuna (Regresyon Örneği)

In [ ]:
def objective_lgbm(trial, X, y):
    # Aranacak parametre uzayını (search space) tanımlıyoruz
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000, step=20),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    
    model = lgb.LGBMRegressor(**param)
    
    # 5-Fold Cross Validation ile skoru hesapla
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    
    # neg_root_mean_squared_error negatif döner, RMSE için eksi ile çarpıyoruz.
    # Biz hatayı MINIMIZE etmeye çalışıyoruz.
    return -scores.mean()

def run_optuna_lgbm(X, y, n_trials=50):
    """
    Optuna'yı çalıştırır ve en iyi parametreleri döner.
    """
    # Hedefimiz hatayı minimize etmek
    study = optuna.create_study(direction='minimize', study_name="LGBM_Optimization")
    
    # Lambda fonksiyonu ile X ve y'yi içeri aktarıyoruz
    study.optimize(lambda trial: objective_lgbm(trial, X, y), n_trials=n_trials)
    
    print("En iyi deneme:")
    trial = study.best_trial
    print("  Değer (RMSE): ", trial.value)
    print("  Parametreler: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
    return study.best_params

# KULLANIM:
# best_lgbm_params = run_optuna_lgbm(X_train, y_train, n_trials=50)
# final_model = lgb.LGBMRegressor(**best_lgbm_params, random_state=42)
# final_model.fit(X_train, y_train)

## Optuna Görselleştirme (Hangi Parametre Önemli?)

In [ ]:
def plot_optuna_results(study):
    """
    Bu görseller Optuna'nın deneme geçmişini ve parametre önemlerini anlamak için harikadır.
    """
    # Optimizasyon tarihçesi
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()
    
    # Hangi parametrenin skoru daha çok etkilediği
    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.show()

# KULLANIM:
# plot_optuna_results(study)